In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# highest level of qualification by sex
# To download original dataset go to - 
# https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019
deprivation_2019_csv = r"N:\Geodatabase\Raw_Data\Deprivation\File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3(4).csv"
deprivation_2015_csv = r"N:\Geodatabase\Raw_Data\Deprivation\imd2010adj_2015.csv"
population_data_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Population\lsoa2021_2019_lookup_table_with_population_data.csv"

## 2. Input Link to LSOA 2011 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2011-boundaries-ew-bfe-v3/about

In [3]:
lsoa2011_shapefile = r"N:\Geodatabase\Raw_Data\Census 2011\Boundaries\Lower_Layer_Super_Output_Areas_Dec_2011_Boundaries_Full_Extent_BFE_EW_V3_2022_4117212475924899368(1)\LSOA_2011_EW_BFE_V3.shp"

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [4]:
deprivation_2019_df = pd.read_csv(deprivation_2019_csv)

rename_mapping = {
    'LSOA code (2011)': 'lsoa11cd',
    'LSOA name (2011)': 'lsoa11nm',
    'Local Authority District code (2019)': 'lad19cd',
    'Local Authority District name (2019)': 'lad19nm',
    'Index of Multiple Deprivation (IMD) Score': 'index_of_multiple_deprivation_score',
    'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)': 'index_of_multiple_deprivation_rank',
    'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)': 'index_of_multiple_deprivation_decile',
    'Income Score (rate)': 'income_score',
    'Income Rank (where 1 is most deprived)': 'income_rank',
    'Income Decile (where 1 is most deprived 10% of LSOAs)': 'income_decile',
    'Employment Score (rate)': 'employment_score',
    'Employment Rank (where 1 is most deprived)': 'employment_rank',
    'Employment Decile (where 1 is most deprived 10% of LSOAs)': 'employment_decile',
    'Education, Skills and Training Score': 'education_skills_and_training_score',
    'Education, Skills and Training Rank (where 1 is most deprived)': 'education_skills_and_training_rank',
    'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)': 'education_skills_and_training_decile',
    'Health Deprivation and Disability Score': 'health_deprivation_and_disability_score',
    'Health Deprivation and Disability Rank (where 1 is most deprived)': 'health_deprivation_and_disability_rank',
    'Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)': 'health_deprivation_and_disability_decile',
    'Crime Score': 'crime_score',
    'Crime Rank (where 1 is most deprived)': 'crime_rank',
    'Crime Decile (where 1 is most deprived 10% of LSOAs)': 'crime_decile',
    'Barriers to Housing and Services Score': 'barriers_to_housing_and_services_score',
    'Barriers to Housing and Services Rank (where 1 is most deprived)': 'barriers_to_housing_and_services_rank',
    'Barriers to Housing and Services Decile (where 1 is most deprived 10% of LSOAs)': 'barriers_to_housing_and_services_decile',
    'Living Environment Score': 'living_environment_score',
    'Living Environment Rank (where 1 is most deprived)': 'living_environment_rank',
    'Living Environment Decile (where 1 is most deprived 10% of LSOAs)': 'living_environment_decile',
    'Income Deprivation Affecting Children Index (IDACI) Score (rate)': 'income_deprivation_affecting_children_index_score',
    'Income Deprivation Affecting Children Index (IDACI) Rank (where 1 is most deprived)': 'income_deprivation_affecting_children_index_rank',
    'Income Deprivation Affecting Children Index (IDACI) Decile (where 1 is most deprived 10% of LSOAs)': 'income_deprivation_affecting_children_index_decile',
    'Income Deprivation Affecting Older People (IDAOPI) Score (rate)': 'income_deprivation_affecting_older_people_score',
    'Income Deprivation Affecting Older People (IDAOPI) Rank (where 1 is most deprived)': 'income_deprivation_affecting_older_people_rank',
    'Income Deprivation Affecting Older People (IDAOPI) Decile (where 1 is most deprived 10% of LSOAs)': 'income_deprivation_affecting_older_people_decile',
    'Children and Young People Sub-domain Score': 'children_and_young_people_sub_domain_score',
    'Children and Young People Sub-domain Rank (where 1 is most deprived)': 'children_and_young_people_sub_domain_rank',
    'Children and Young People Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'children_and_young_people_sub_domain_decile',
    'Adult Skills Sub-domain Score': 'adult_skills_sub_domain_score',
    'Adult Skills Sub-domain Rank (where 1 is most deprived)': 'adult_skills_sub_domain_rank',
    'Adult Skills Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'adult_skills_sub_domain_decile',
    'Geographical Barriers Sub-domain Score': 'geographical_barriers_sub_domain_score',
    'Geographical Barriers Sub-domain Rank (where 1 is most deprived)': 'geographical_barriers_sub_domain_rank',
    'Geographical Barriers Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'geographical_barriers_sub_domain_decile',
    'Wider Barriers Sub-domain Score': 'wider_barriers_sub_domain_score',
    'Wider Barriers Sub-domain Rank (where 1 is most deprived)': 'wider_barriers_sub_domain_rank',
    'Wider Barriers Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'wider_barriers_sub_domain_decile',
    'Indoors Sub-domain Score': 'indoors_sub_domain_score',
    'Indoors Sub-domain Rank (where 1 is most deprived)': 'indoors_sub_domain_rank',
    'Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'indoors_sub_domain_decile',
    'Outdoors Sub-domain Score': 'outdoors_sub_domain_score',
    'Outdoors Sub-domain Rank (where 1 is most deprived)': 'outdoors_sub_domain_rank',
    'Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs)': 'outdoors_sub_domain_decile',
    'Total population: mid 2015 (excluding prisoners)': 'total_population_mid_2015',
    'Dependent Children aged 0-15: mid 2015 (excluding prisoners)': 'dependent_children_aged_0_15_mid_2015',
    'Population aged 16-59: mid 2015 (excluding prisoners)': 'population_aged_16_59_mid_2015',
    'Older population aged 60 and over: mid 2015 (excluding prisoners)': 'older_population_aged_60_and_over_mid_2015',
    'Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners) ': 'working_age_population_18_59_64'
}

deprivation_2019_df.rename(columns = rename_mapping, inplace = True)
deprivation_2019_df.drop(['total_population_mid_2015','dependent_children_aged_0_15_mid_2015','population_aged_16_59_mid_2015','older_population_aged_60_and_over_mid_2015','working_age_population_18_59_64'],1,inplace = True)
deprivation_2019_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_21084\1452111051.py:64: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  deprivation_2019_df.drop(['total_population_mid_2015','dependent_children_aged_0_15_mid_2015','population_aged_16_59_mid_2015','older_population_aged_60_and_over_mid_2015','working_age_population_18_59_64'],1,inplace = True)


,lsoa11cd,lsoa11nm,lad19cd,lad19nm,index_of_multiple_deprivation_score,index_of_multiple_deprivation_rank,index_of_multiple_deprivation_decile,income_score,income_rank,income_decile,employment_score,employment_rank,employment_decile,education_skills_and_training_score,education_skills_and_training_rank,education_skills_and_training_decile,health_deprivation_and_disability_score,health_deprivation_and_disability_rank,health_deprivation_and_disability_decile,crime_score,crime_rank,crime_decile,barriers_to_housing_and_services_score,barriers_to_housing_and_services_rank,barriers_to_housing_and_services_decile,living_environment_score,living_environment_rank,living_environment_decile,income_deprivation_affecting_children_index_score,income_deprivation_affecting_children_index_rank,income_deprivation_affecting_children_index_decile,income_deprivation_affecting_older_people_score,income_deprivation_affecting_older_people_rank,income_deprivation_affecting_older_people_decile,children_and_young_people_sub_domain_score,children_and_young_people_sub_domain_rank,children_and_young_people_sub_domain_decile,adult_skills_sub_domain_score,adult_skills_sub_domain_rank,adult_skills_sub_domain_decile,geographical_barriers_sub_domain_score,geographical_barriers_sub_domain_rank,geographical_barriers_sub_domain_decile,wider_barriers_sub_domain_score,wider_barriers_sub_domain_rank,wider_barriers_sub_domain_decile,indoors_sub_domain_score,indoors_sub_domain_rank,indoors_sub_domain_decile,outdoors_sub_domain_score,outdoors_sub_domain_rank,outdoors_sub_domain_decile
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,0.010,32742,10,0.024,32842,10,-1.654,32113,10,-2.012,32662,10,29.472,7319,3,31.873,7789,3,0.006,32806,10,0.012,32820,10,-2.107,32777,10,0.032,32843,10,-0.430,22985,7,3.587,3216,1,0.006,16364,5,1.503,1615,1
1,E01000002,City of London 001B,E09000001,City of London,5.143,30379,10,0.034,29901,10,0.027,31190,10,0.063,32832,10,-1.115,29705,10,-2.343,32789,10,24.412,11707,4,23.084,13070,4,0.037,29682,10,0.030,31938,10,-1.907,32666,10,0.034,32841,10,-1.060,30223,10,3.231,3894,2,-0.410,22676,7,1.196,2969,1
2,E01000003,City of London 001C,E09000001,City of London,19.402,14915,5,0.086,18510,6,0.086,15103,5,5.804,26386,9,-0.102,17600,6,-1.032,29363,9,40.103,2157,1,40.535,4092,2,0.052,27063,9,0.128,16377,5,-0.292,20837,7,0.142,30999,10,-0.691,26719,9,5.173,818,1,-0.054,17318,6,2.207,162,1
3,E01000005,City of London 001E,E09000001,City of London,28.652,8678,3,0.211,6029,2,0.136,7833,3,22.260,12370,4,-0.121,17907,6,-1.317,31059,10,39.900,2217,1,28.979,9397,3,0.209,9458,3,0.322,3885,2,0.338,10914,4,0.321,13658,5,-1.167,30865,10,5.361,672,1,-0.604,25218,8,1.769,849,1
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,19.837,14486,5,0.117,14023,5,0.059,21692,7,14.798,17511,6,-0.359,21581,7,-0.147,18848,6,45.171,1033,1,26.888,10629,4,0.155,13592,5,0.162,12934,4,-0.366,21932,7,0.325,13244,5,-0.400,22527,7,5.590,520,1,0.110,14745,5,0.969,4368,2


In [5]:
deprivation_2015_df = pd.read_csv(deprivation_2015_csv)

columns_to_keep = ['LSOA code (2011)',
 '2010imd_rank',
 '2015Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)',
 '2015Income Rank (where 1 is most deprived)',
 '2015Employment Rank (where 1 is most deprived)',
 '2015Education, Skills and Training Rank (where 1 is most deprived)',
 '2015Health Deprivation and Disability Rank (where 1 is most deprived)',
 '2015Crime Rank (where 1 is most deprived)',
 '2015Barriers to Housing and Services Rank (where 1 is most deprived)',
 '2015Living Environment Rank (where 1 is most deprived)',
 '2015Income Deprivation Affecting Children Index (IDACI) Rank (where 1 is most deprived)',
 '2015Income Deprivation Affecting Older People (IDAOPI) Rank (where 1 is most deprived)',
 '2015Children and Young People Sub-domain Rank (where 1 is most deprived)',
 '2015Adult Skills Sub-domain Rank (where 1 is most deprived)',
 '2015Geographical Barriers Sub-domain Rank (where 1 is most deprived)',
 '2015Wider Barriers Sub-domain Rank (where 1 is most deprived)',
 '2015Indoors Sub-domain Rank (where 1 is most deprived)',
 '2015Outdoors Sub-domain Rank (where 1 is most deprived)'
]
 
deprivation_2015_df = deprivation_2015_df[columns_to_keep]

rename_mapping_v2 = {
    'LSOA code (2011)': 'lsoa11cd',
    '2010imd_rank': 'index_of_multiple_deprivation_rank_2010',
    '2015Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)': 'index_of_multiple_deprivation_rank_2015',
    '2015Income Rank (where 1 is most deprived)': 'income_rank_2015',
    '2015Employment Rank (where 1 is most deprived)': 'employment_rank_2015',
    '2015Education, Skills and Training Rank (where 1 is most deprived)': 'education_skills_and_training_rank_2015',
    '2015Health Deprivation and Disability Rank (where 1 is most deprived)': 'health_deprivation_and_disability_rank_2015',
    '2015Crime Rank (where 1 is most deprived)': 'crime_rank_2015',
    '2015Barriers to Housing and Services Rank (where 1 is most deprived)': 'barriers_to_housing_and_services_rank_2015',
    '2015Living Environment Rank (where 1 is most deprived)': 'living_environment_rank_2015',
    '2015Income Deprivation Affecting Children Index (IDACI) Rank (where 1 is most deprived)': 'income_deprivation_affecting_children_index_rank_2015',
    '2015Income Deprivation Affecting Older People (IDAOPI) Rank (where 1 is most deprived)': 'income_deprivation_affecting_older_people_rank_2015',
    '2015Children and Young People Sub-domain Rank (where 1 is most deprived)': 'children_and_young_people_sub_domain_rank_2015',
    '2015Adult Skills Sub-domain Rank (where 1 is most deprived)': 'adult_skills_sub_domain_rank_2015',
    '2015Geographical Barriers Sub-domain Rank (where 1 is most deprived)': 'geographical_barriers_sub_domain_rank_2015',
    '2015Wider Barriers Sub-domain Rank (where 1 is most deprived)': 'wider_barriers_sub_domain_rank_2015',
    '2015Indoors Sub-domain Rank (where 1 is most deprived)': 'indoors_sub_domain_rank_2015',
    '2015Outdoors Sub-domain Rank (where 1 is most deprived)': 'outdoors_sub_domain_rank_2015'
}

deprivation_2015_df.rename(columns = rename_mapping_v2, inplace = True)

deprivation_2015_df.head()

,lsoa11cd,index_of_multiple_deprivation_rank_2010,index_of_multiple_deprivation_rank_2015,income_rank_2015,employment_rank_2015,education_skills_and_training_rank_2015,health_deprivation_and_disability_rank_2015,crime_rank_2015,barriers_to_housing_and_services_rank_2015,living_environment_rank_2015,income_deprivation_affecting_children_index_rank_2015,income_deprivation_affecting_older_people_rank_2015,children_and_young_people_sub_domain_rank_2015,adult_skills_sub_domain_rank_2015,geographical_barriers_sub_domain_rank_2015,wider_barriers_sub_domain_rank_2015,indoors_sub_domain_rank_2015,outdoors_sub_domain_rank_2015
0,E01000001,29130,29111,32813,32716,32840,32408,32360,9056,6750,32817,32770,32785,32843,19883,4720,10915,2750
1,E01000002,29765,28855,32780,32692,32836,32431,32436,9716,5711,32832,32415,32733,32841,19444,5201,6550,5551
2,E01000003,20251,14621,17600,18133,22251,11639,29256,5633,2593,15804,15417,14843,30999,22529,2591,7531,633
3,E01000005,13109,10285,7255,9641,17420,15392,27804,3981,6620,7874,2525,20814,13658,31396,1214,14702,1287
4,E01000006,16582,12345,14190,16938,17806,21793,5849,2197,8440,15692,11114,22470,13244,15202,1807,8776,7733


In [6]:
combined_df = pd.merge(deprivation_2019_df,deprivation_2015_df, how = 'left', on = 'lsoa11cd')
combined_df['imd_rank_change_2010_2015'] = combined_df['index_of_multiple_deprivation_rank_2015'] - combined_df['index_of_multiple_deprivation_rank_2010'] 
combined_df['imd_rank_change_2015_2019'] = combined_df['index_of_multiple_deprivation_rank'] - combined_df['index_of_multiple_deprivation_rank_2015'] 

def classify_imd_rank_trend(row):
    change_1 = row['imd_rank_change_2010_2015']
    change_2 = row['imd_rank_change_2015_2019']
    
    if change_1 > 0 and change_2 > 0:
        return 'Consistently improved'
    elif change_1 < 0 and change_2 < 0:
        return 'Consistently worsened'
    elif change_1 > 0 and change_2 < 0:
        return 'Improved then worsened'
    elif change_1 < 0 and change_2 > 0:
        return 'Worsened then improved'
    else:
        return 'No clear trend'

combined_df['imd_rank_trend_2010_2019'] = combined_df.apply(classify_imd_rank_trend, axis=1)
combined_df.head()


,lsoa11cd,lsoa11nm,lad19cd,lad19nm,index_of_multiple_deprivation_score,index_of_multiple_deprivation_rank,index_of_multiple_deprivation_decile,income_score,income_rank,income_decile,employment_score,employment_rank,employment_decile,education_skills_and_training_score,education_skills_and_training_rank,education_skills_and_training_decile,health_deprivation_and_disability_score,health_deprivation_and_disability_rank,health_deprivation_and_disability_decile,crime_score,crime_rank,crime_decile,barriers_to_housing_and_services_score,barriers_to_housing_and_services_rank,barriers_to_housing_and_services_decile,living_environment_score,living_environment_rank,living_environment_decile,income_deprivation_affecting_children_index_score,income_deprivation_affecting_children_index_rank,income_deprivation_affecting_children_index_decile,income_deprivation_affecting_older_people_score,income_deprivation_affecting_older_people_rank,income_deprivation_affecting_older_people_decile,children_and_young_people_sub_domain_score,children_and_young_people_sub_domain_rank,children_and_young_people_sub_domain_decile,adult_skills_sub_domain_score,adult_skills_sub_domain_rank,adult_skills_sub_domain_decile,geographical_barriers_sub_domain_score,geographical_barriers_sub_domain_rank,geographical_barriers_sub_domain_decile,wider_barriers_sub_domain_score,wider_barriers_sub_domain_rank,wider_barriers_sub_domain_decile,indoors_sub_domain_score,indoors_sub_domain_rank,indoors_sub_domain_decile,outdoors_sub_domain_score,outdoors_sub_domain_rank,outdoors_sub_domain_decile,index_of_multiple_deprivation_rank_2010,index_of_multiple_deprivation_rank_2015,income_rank_2015,employment_rank_2015,education_skills_and_training_rank_2015,health_deprivation_and_disability_rank_2015,crime_rank_2015,barriers_to_housing_and_services_rank_2015,living_environment_rank_2015,income_deprivation_affecting_children_index_rank_2015,income_deprivation_affecting_older_people_rank_2015,children_and_young_people_sub_domain_rank_2015,adult_skills_sub_domain_rank_2015,geographical_barriers_sub_domain_rank_2015,wider_barriers_sub_domain_rank_2015,indoors_sub_domain_rank_2015,outdoors_sub_domain_rank_2015,imd_rank_change_2010_2015,imd_rank_change_2015_2019,imd_rank_trend_2010_2019
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,0.010,32742,10,0.024,32842,10,-1.654,32113,10,-2.012,32662,10,29.472,7319,3,31.873,7789,3,0.006,32806,10,0.012,32820,10,-2.107,32777,10,0.032,32843,10,-0.430,22985,7,3.587,3216,1,0.006,16364,5,1.503,1615,1,29130,29111,32813,32716,32840,32408,32360,9056,6750,32817,32770,32785,32843,19883,4720,10915,2750,-19,88,Worsened then improved
1,E01000002,City of London 001B,E09000001,City of London,5.143,30379,10,0.034,29901,10,0.027,31190,10,0.063,32832,10,-1.115,29705,10,-2.343,32789,10,24.412,11707,4,23.084,13070,4,0.037,29682,10,0.030,31938,10,-1.907,32666,10,0.034,32841,10,-1.060,30223,10,3.231,3894,2,-0.410,22676,7,1.196,2969,1,29765,28855,32780,32692,32836,32431,32436,9716,5711,32832,32415,32733,32841,19444,5201,6550,5551,-910,1524,Worsened then improved
2,E01000003,City of London 001C,E09000001,City of London,19.402,14915,5,0.086,18510,6,0.086,15103,5,5.804,26386,9,-0.102,17600,6,-1.032,29363,9,40.103,2157,1,40.535,4092,2,0.052,27063,9,0.128,16377,5,-0.292,20837,7,0.142,30999,10,-0.691,26719,9,5.173,818,1,-0.054,17318,6,2.207,162,1,20251,14621,17600,18133,22251,11639,29256,5633,2593,15804,15417,14843,30999,22529,2591,7531,633,-5630,294,Worsened then improved
3,E01000005,City of London 001E,E09000001,City of London,28.652,8678,3,0.211,6029,2,0.136,7833,3,22.260,12370,4,-0.121,17907,6,-1.317,31059,10,39.900,2217,1,28.979,9397,3,0.209,9458,3,0.322,3885,2,0.338,10914,4,0.321,13658,5,-1.167,30865,10,5.361,672,1,-0.604,25218,8,1.769,849,1,13109,10285,7255,9641,17420,15392,27804,3981,6620,7874,2525,20814,13658,31396,1214,14702,1287,-2824,-1607,Consistently worsened
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,19.

In [8]:
def classify_2015_2019_trend(change):
    if change > 0:
        return 'Improved'
    elif change < 0:
        return 'Worsened'
    else:
        return 'No change'
    
combined_df['income_rank_change_2015_2019'] = combined_df['income_rank'] - combined_df['income_rank_2015'] 
combined_df['employment_rank_change_2015_2019'] = combined_df['employment_rank'] - combined_df['employment_rank_2015'] 
combined_df['education_rank_change_2015_2019'] = combined_df['education_skills_and_training_rank'] - combined_df['education_skills_and_training_rank_2015'] 
combined_df['health_rank_change_2015_2019'] = combined_df['health_deprivation_and_disability_rank'] - combined_df['health_deprivation_and_disability_rank_2015'] 
combined_df['crime_rank_change_2015_2019'] = combined_df['crime_rank'] - combined_df['crime_rank_2015'] 
combined_df['barriers_to_housing_rank_change_2015_2019'] = combined_df['barriers_to_housing_and_services_rank'] - combined_df['barriers_to_housing_and_services_rank_2015'] 
combined_df['living_environment_rank_change_2015_2019'] = combined_df['living_environment_rank'] - combined_df['living_environment_rank_2015'] 
combined_df['income_affecting_children_rank_change_2015_2019'] = combined_df['income_deprivation_affecting_children_index_rank'] - combined_df['income_deprivation_affecting_children_index_rank_2015'] 
combined_df['income_affecting_elderly_rank_change_2015_2019'] = combined_df['income_deprivation_affecting_older_people_rank'] - combined_df['income_deprivation_affecting_older_people_rank_2015'] 
combined_df['children_and_young_people_rank_change_2015_2019'] = combined_df['children_and_young_people_sub_domain_rank'] - combined_df['children_and_young_people_sub_domain_rank_2015'] 
combined_df['adult_skills_rank_change_2015_2019'] = combined_df['adult_skills_sub_domain_rank'] - combined_df['adult_skills_sub_domain_rank_2015'] 
combined_df['geographical_barriers_rank_change_2015_2019'] = combined_df['geographical_barriers_sub_domain_rank'] - combined_df['geographical_barriers_sub_domain_rank_2015'] 
combined_df['wider_barriers_rank_change_2015_2019'] = combined_df['wider_barriers_sub_domain_rank'] - combined_df['wider_barriers_sub_domain_rank_2015'] 
combined_df['indoors_rank_change_2015_2019'] = combined_df['indoors_sub_domain_rank'] - combined_df['indoors_sub_domain_rank_2015'] 
combined_df['outdoors_rank_change_2015_2019'] = combined_df['outdoors_sub_domain_rank'] - combined_df['outdoors_sub_domain_rank_2015'] 

    
columns_to_classify = [
    'income_rank_change_2015_2019',
    'employment_rank_change_2015_2019',
    'education_rank_change_2015_2019',
    'health_rank_change_2015_2019',
    'crime_rank_change_2015_2019',
    'barriers_to_housing_rank_change_2015_2019',
    'living_environment_rank_change_2015_2019',
    'income_affecting_children_rank_change_2015_2019',
    'income_affecting_elderly_rank_change_2015_2019',
    'children_and_young_people_rank_change_2015_2019',
    'adult_skills_rank_change_2015_2019',
    'geographical_barriers_rank_change_2015_2019',
    'wider_barriers_rank_change_2015_2019',
    'indoors_rank_change_2015_2019',
    'outdoors_rank_change_2015_2019'
]

for col in columns_to_classify:
    trend_col = col.replace('_change_2015_2019', '_trend_2015_2019')
    combined_df[trend_col] = combined_df[col].apply(classify_2015_2019_trend)
    
combined_df.head()

,lsoa11cd,lsoa11nm,lad19cd,lad19nm,index_of_multiple_deprivation_score,index_of_multiple_deprivation_rank,index_of_multiple_deprivation_decile,income_score,income_rank,income_decile,employment_score,employment_rank,employment_decile,education_skills_and_training_score,education_skills_and_training_rank,education_skills_and_training_decile,health_deprivation_and_disability_score,health_deprivation_and_disability_rank,health_deprivation_and_disability_decile,crime_score,crime_rank,crime_decile,barriers_to_housing_and_services_score,barriers_to_housing_and_services_rank,barriers_to_housing_and_services_decile,living_environment_score,living_environment_rank,living_environment_decile,income_deprivation_affecting_children_index_score,income_deprivation_affecting_children_index_rank,income_deprivation_affecting_children_index_decile,income_deprivation_affecting_older_people_score,income_deprivation_affecting_older_people_rank,income_deprivation_affecting_older_people_decile,children_and_young_people_sub_domain_score,children_and_young_people_sub_domain_rank,children_and_young_people_sub_domain_decile,adult_skills_sub_domain_score,adult_skills_sub_domain_rank,adult_skills_sub_domain_decile,geographical_barriers_sub_domain_score,geographical_barriers_sub_domain_rank,geographical_barriers_sub_domain_decile,wider_barriers_sub_domain_score,wider_barriers_sub_domain_rank,wider_barriers_sub_domain_decile,indoors_sub_domain_score,indoors_sub_domain_rank,indoors_sub_domain_decile,outdoors_sub_domain_score,outdoors_sub_domain_rank,outdoors_sub_domain_decile,index_of_multiple_deprivation_rank_2010,index_of_multiple_deprivation_rank_2015,income_rank_2015,employment_rank_2015,education_skills_and_training_rank_2015,health_deprivation_and_disability_rank_2015,crime_rank_2015,barriers_to_housing_and_services_rank_2015,living_environment_rank_2015,income_deprivation_affecting_children_index_rank_2015,income_deprivation_affecting_older_people_rank_2015,children_and_young_people_sub_domain_rank_2015,adult_skills_sub_domain_rank_2015,geographical_barriers_sub_domain_rank_2015,wider_barriers_sub_domain_rank_2015,indoors_sub_domain_rank_2015,outdoors_sub_domain_rank_2015,imd_rank_change_2010_2015,imd_rank_change_2015_2019,imd_rank_trend_2010_2019,income_rank_change_2015_2019,employment_rank_change_2015_2019,education_rank_change_2015_2019,health_rank_change_2015_2019,crime_rank_change_2015_2019,barriers_to_housing_rank_change_2015_2019,living_environment_rank_change_2015_2019,income_affecting_children_rank_change_2015_2019,income_affecting_elderly_rank_change_2015_2019,children_and_young_people_rank_change_2015_2019,adult_skills_rank_change_2015_2019,geographical_barriers_rank_change_2015_2019,wider_barriers_rank_change_2015_2019,indoors_rank_change_2015_2019,outdoors_rank_change_2015_2019,income_rank_trend_2015_2019,employment_rank_trend_2015_2019,education_rank_trend_2015_2019,health_rank_trend_2015_2019,crime_rank_trend_2015_2019,barriers_to_housing_rank_trend_2015_2019,living_environment_rank_trend_2015_2019,income_affecting_children_rank_trend_2015_2019,income_affecting_elderly_rank_trend_2015_2019,children_and_young_people_rank_trend_2015_2019,adult_skills_rank_trend_2015_2019,geographical_barriers_rank_trend_2015_2019,wider_barriers_rank_trend_2015_2019,indoors_rank_trend_2015_2019,outdoors_rank_trend_2015_2019
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,0.010,32742,10,0.024,32842,10,-1.654,32113,10,-2.012,32662,10,29.472,7319,3,31.873,7789,3,0.006,32806,10,0.012,32820,10,-2.107,32777,10,0.032,32843,10,-0.430,22985,7,3.587,3216,1,0.006,16364,5,1.503,1615,1,29130,29111,32813,32716,32840,32408,32360,9056,6750,32817,32770,32785,32843,19883,4720,10915,2750,-19,88,Worsened then improved,18,26,2,-295,302,-1737,1039,-11,50,-8,0,3102,-1504,5449,-1135,Improved,Improved,Improved,Worsened,Improved,Worsened,Improved,Worsened,Improved,Worsened,No change,Improved,Worsened,Improved,Worsened
1,E01000002,City of London 00

In [9]:
population_data_df = pd.read_csv(population_data_csv)
population_data_lsoa11_df = population_data_df.groupby('lsoa11cd').sum(numeric_only=True).reset_index()
population_data_lsoa11_df.head()

,lsoa11cd,total_population_2022,dependent_children_aged_0_15_2022,older_population_aged_65_and_over_2022,working_population_16_64_2022,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_18_and_above_count
0,E01000001,1795.0,149.0,429.0,1217.0,154.0,0.0,70.0,9.0,14.0,37.0,13.0,11.0,75.0
1,E01000002,1671.0,81.0,294.0,1296.0,141.0,0.0,53.0,13.0,21.0,33.0,15.0,6.0,75.0
2,E01000003,1896.0,136.0,321.0,1439.0,137.0,0.0,70.0,9.0,12.0,21.0,9.0,16.0,58.0
3,E01000005,1737.0,177.0,101.0,1459.0,250.0,0.0,89.0,32.0,57.0,47.0,5.0,20.0,129.0
4,E01000006,1837.0,397.0,170.0,1270.0,416.0,0.0,260.0,49.0,50.0,29.0,13.0,15.0,107.0


## 4. Load GIS shapefile 

In [10]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2011boundary_df = gpd.read_file(lsoa2011_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2011boundary_df.head()

Shapefile loaded successfully from the server.


,LSOA11CD,LSOA11NM,BNG_E,BNG_N,LONG_,LAT,Shape_Leng,GlobalID,geometry
0,E01000034,Barking and Dagenham 003A,550689,186433,0.172296,51.5567,6006.847623,2a20725d-7bab-4a96-b09b-154eb16073a5,"POLYGON ((550783.502 186822.004, 550783.084 18..."
1,E01000035,Barking and Dagenham 010A,549957,185575,0.161380,51.5491,3609.905134,47eef4c6-f44f-43c7-ba2f-bd8a4e62b5ee,"POLYGON ((550266.910 185957.709, 550269.100 18..."
2,E01000036,Barking and Dagenham 010B,550645,185231,0.171148,51.5459,4393.928508,57bf633a-6b2d-49f1-87fa-e64b1c2503b3,"POLYGON ((549843.501 185417.140, 549850.000 18..."
3,E01000037,Barking and Dagenham 003B,551195,187087,0.179870,51.5624,2905.634678,de173b53-0b6e-4886-a4b6-3b59153cfcf3,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E01000038,Barking and Dagenham 003C,550700,187149,0.172761,51.5631,2305.558903,16d8a8a9-39a1-40b8-a459-41b2f1fd0d04,"POLYGON ((550920.362 187341.138, 550921.876 18..."


## 5. GIS data management

### Remove Rename fields

In [ ]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2011boundary_df = gpd.read_file(lsoa2011_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2011boundary_df.head()

In [11]:
#Drop and rename fields
lsoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)
lsoa2011boundary_df.rename(columns = {'LSOA11CD':'lsoa11cd','LSOA11NM':'lsoa11nm'}, inplace = True)
lsoa2011boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_21084\249820046.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)


,lsoa11cd,lsoa11nm,geometry
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18..."
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18..."
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18..."
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18..."


### Combine with boundary lookup table

#### LSOA19 to LSOA21 to LAD22

In [12]:
lsoa19lookup = pd.read_csv(r"N:\Geodatabase\Raw_Data\Look up tables\LSOA_(2011)_to_LSOA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_for_EW_(V3)(1).csv")
lsoa19lookup = lsoa19lookup[['LSOA11CD','LSOA21CD','LSOA21NM','LAD22CD','LAD22NM']]
lsoa19lookup.rename(columns = {'LSOA11CD':'lsoa11cd','LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm','LAD22CD':'lad22cd','LAD22NM':'lad22nm'}, inplace = True)
lsoa19lookup.head()

,lsoa11cd,lsoa21cd,lsoa21nm,lad22cd,lad22nm
0,E01031349,E01031349,Adur 001A,E07000223,Adur
1,E01031350,E01031350,Adur 001B,E07000223,Adur
2,E01031351,E01031351,Adur 001C,E07000223,Adur
3,E01031352,E01031352,Adur 001D,E07000223,Adur
4,E01031370,E01031370,Adur 001E,E07000223,Adur


In [13]:
lsoa2011boundary_df = pd.merge(lsoa2011boundary_df, lsoa19lookup, how = 'left', on = 'lsoa11cd')
lsoa2011boundary_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham


#### LAD22 to REGION22

In [14]:
# Define the file path for lad to region
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_21084\2710224882.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [15]:
lsoa2011boundary_df = pd.merge(lsoa2011boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2011boundary_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham,E12000007,London
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham,E12000007,London
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham,E12000007,London
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham,E12000007,London


### Add data management fields

In [16]:
# Add new data management fields with specified values
lsoa2011boundary_df['data_source'] = "Ministry of Housing, Communities & Local Government"
lsoa2011boundary_df['data_resolution'] = "LSOA 2011"
lsoa2011boundary_df['data_time_period'] = "2010-2019"
lsoa2011boundary_df['data_web_link'] = "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019"
lsoa2011boundary_df.head()

,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...


### Update Area

In [17]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2011boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2011boundary_df['area_ha'] = lsoa2011boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2011boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2011boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,lsoa11cd,lsoa11nm,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...,51.365911
1,E01000035,Barking and Dagenham 010A,"POLYGON ((550266.910 185957.709, 550269.100 18...",E01000035,Barking and Dagenham 010A,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...,39.716142
2,E01000036,Barking and Dagenham 010B,"POLYGON ((549843.501 185417.140, 549850.000 18...",E01000036,Barking and Dagenham 010B,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...,41.868830
3,E01000037,Barking and Dagenham 003B,"POLYGON ((551550.056 187364.705, 551528.633 18...",E01000037,Barking and Dagenham 003B,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...,23.343335
4,E01000038,Barking and Dagenham 003C,"POLYGON ((550920.362 187341.138, 550921.876 18...",E01000038,Barking and Dagenham 003C,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA 2011,2010-2019,https://www.gov.uk/government/statistics/engli...,21.404577


# 7. Combine Geometry and data

In [18]:
mhclg_deprivation_lsoa2011_gdb_df = pd.merge(lsoa2011boundary_df,combined_df, how = 'left', on='lsoa11cd')
mhclg_deprivation_lsoa2011_gdb_df = pd.merge(mhclg_deprivation_lsoa2011_gdb_df,population_data_lsoa11_df, how = 'left', on='lsoa11cd')
mhclg_deprivation_lsoa2011_gdb_df.head()

,lsoa11cd,lsoa11nm_x,geometry,lsoa21cd,lsoa21nm,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,lsoa11nm_y,lad19cd,lad19nm,index_of_multiple_deprivation_score,index_of_multiple_deprivation_rank,index_of_multiple_deprivation_decile,income_score,income_rank,income_decile,employment_score,employment_rank,employment_decile,education_skills_and_training_score,education_skills_and_training_rank,education_skills_and_training_decile,health_deprivation_and_disability_score,health_deprivation_and_disability_rank,health_deprivation_and_disability_decile,crime_score,crime_rank,crime_decile,barriers_to_housing_and_services_score,barriers_to_housing_and_services_rank,barriers_to_housing_and_services_decile,living_environment_score,living_environment_rank,living_environment_decile,income_deprivation_affecting_children_index_score,income_deprivation_affecting_children_index_rank,income_deprivation_affecting_children_index_decile,income_deprivation_affecting_older_people_score,income_deprivation_affecting_older_people_rank,income_deprivation_affecting_older_people_decile,children_and_young_people_sub_domain_score,children_and_young_people_sub_domain_rank,children_and_young_people_sub_domain_decile,adult_skills_sub_domain_score,adult_skills_sub_domain_rank,adult_skills_sub_domain_decile,geographical_barriers_sub_domain_score,geographical_barriers_sub_domain_rank,geographical_barriers_sub_domain_decile,wider_barriers_sub_domain_score,wider_barriers_sub_domain_rank,wider_barriers_sub_domain_decile,indoors_sub_domain_score,indoors_sub_domain_rank,indoors_sub_domain_decile,outdoors_sub_domain_score,outdoors_sub_domain_rank,outdoors_sub_domain_decile,index_of_multiple_deprivation_rank_2010,index_of_multiple_deprivation_rank_2015,income_rank_2015,employment_rank_2015,education_skills_and_training_rank_2015,health_deprivation_and_disability_rank_2015,crime_rank_2015,barriers_to_housing_and_services_rank_2015,living_environment_rank_2015,income_deprivation_affecting_children_index_rank_2015,income_deprivation_affecting_older_people_rank_2015,children_and_young_people_sub_domain_rank_2015,adult_skills_sub_domain_rank_2015,geographical_barriers_sub_domain_rank_2015,wider_barriers_sub_domain_rank_2015,indoors_sub_domain_rank_2015,outdoors_sub_domain_rank_2015,imd_rank_change_2010_2015,imd_rank_change_2015_2019,imd_rank_trend_2010_2019,income_rank_change_2015_2019,employment_rank_change_2015_2019,education_rank_change_2015_2019,health_rank_change_2015_2019,crime_rank_change_2015_2019,barriers_to_housing_rank_change_2015_2019,living_environment_rank_change_2015_2019,income_affecting_children_rank_change_2015_2019,income_affecting_elderly_rank_change_2015_2019,children_and_young_people_rank_change_2015_2019,adult_skills_rank_change_2015_2019,geographical_barriers_rank_change_2015_2019,wider_barriers_rank_change_2015_2019,indoors_rank_change_2015_2019,outdoors_rank_change_2015_2019,income_rank_trend_2015_2019,employment_rank_trend_2015_2019,education_rank_trend_2015_2019,health_rank_trend_2015_2019,crime_rank_trend_2015_2019,barriers_to_housing_rank_trend_2015_2019,living_environment_rank_trend_2015_2019,income_affecting_children_rank_trend_2015_2019,income_affecting_elderly_rank_trend_2015_2019,children_and_young_people_rank_trend_2015_2019,adult_skills_rank_trend_2015_2019,geographical_barriers_rank_trend_2015_2019,wider_barriers_rank_trend_2015_2019,indoors_rank_trend_2015_2019,outdoors_rank_trend_2015_2019,total_population_2022,dependent_children_aged_0_15_2022,older_population_aged_65_and_over_2022,working_population_16_64_2022,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_18_and_above_count
0,E01000034,Barking and Dagenham 003A,"POLYGON ((550783.502 186822.004, 550783.084 18...",E01000034,Barking and Dagenham 003A,E09000002,Barking and Dagenham,E12000007,London,"Ministry of Housing, Communities & Local Gover...",LSOA

In [19]:
mhclg_deprivation_lsoa2011_gdb_df = mhclg_deprivation_lsoa2011_gdb_df.drop_duplicates(subset='lsoa11cd', keep='first')

# 9. Upload to geodatabase

In [20]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "mhclg_index_of_multiple_deprivation"  # Desired table name
primary_key_column = "msoa11cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [21]:
# Ensure the GeoDataFrame has a valid CRS before writing
if mhclg_deprivation_lsoa2011_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    mhclg_deprivation_lsoa2011_gdb_df.set_crs(epsg=27700, inplace=True)

In [22]:
# Publish the GeoDataFrame to PostGIS
mhclg_deprivation_lsoa2011_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.mhclg_index_of_multiple_deprivation
Could not set primary key. It may already exist. Error: (psycopg2.errors.UndefinedColumn) column "msoa11cd" of relation "mhclg_index_of_multiple_deprivation" does not exist

[SQL: 
        ALTER TABLE uk_new.mhclg_index_of_multiple_deprivation
        ADD CONSTRAINT mhclg_index_of_multiple_deprivation_pkey PRIMARY KEY (msoa11cd);
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)
Could not create spatial index. It may already exist. Error: (psycopg2.errors.InFailedSqlTransaction) current transaction is aborted, commands ignored until end of transaction block

[SQL: 
        CREATE INDEX mhclg_index_of_multiple_deprivation_geom_idx
        ON uk_new.mhclg_index_of_multiple_deprivation
        USING GIST (geometry);
    ]
(Background on this error at: https://sqlalche.me/e/20/2j85)
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.mhclg_index_of_multip